In [1]:
import os
import requests
import json
from typing import Optional, Dict, Any
import pandas as pd
from dotenv import load_dotenv

from utils import DuckdbUtils
from dags.data_collector import CryptoDataCollector

1. raw layer

In [2]:
sample_asset = 'BTC'

sample_data_collector = CryptoDataCollector(
    sample_asset,
    'iceberg.cryptocurrencies_project_raw.btc',
    'iceberg.cryptocurrencies_project_raw_stg.btc',
)

     id           descr   report_dt
0  1115  spark job test  2025-12-31
1  1115  spark job test  2025-12-31
2     1           tgete  2025-02-01
3     3            vdsv  2025-06-01
4     1           tgete  2025-02-01
5     3            vdsv  2025-06-01


In [4]:
sample_result_row_cnt = sample_data_collector.process()

/opt/workspace/ETL/dags/data_collector.py:54: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  pdf = pd.read_json(json_data_str, orient='index')


TransactionException: TransactionContext Error: Failed to commit: Failed to commit Iceberg transaction: HTTP GET error on 'http://ozone-s3g:9878/mybucket/warehouse/cryptocurrencies_project_raw_stg/btc/metadata/96b1e69a-787a-4feb-9aed-de9359d36bbf-m0.avro' (HTTP 403)

Authentication Failure - this is usually caused by invalid or missing credentials.
* No credentials are provided.
* See https://duckdb.org/docs/stable/extensions/httpfs/s3api.html

In [ ]:
print(f"Successfully loaded {sample_result_row_cnt} rows")

In [ ]:
SUPPORTED_ASSETS = {
    'BTC': 'BITCOIN',
    'ETH': 'ETHEREUM',
}
ALPHA_VANTAGE_URL = 'https://www.alphavantage.co/query'
load_dotenv() # dotenv_path='../.env'
API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY") # must be specified in your /examples/ETL/.env file

In [5]:
api_params = {
    'function': 'DIGITAL_CURRENCY_DAILY',
    'symbol': sample_asset,
    'market': "USD",
    'apikey': API_KEY,
}

response = requests.get(ALPHA_VANTAGE_URL, params=api_params)

In [6]:
response.raise_for_status()

In [7]:
data = response.json()

In [20]:
import duckdb

In [15]:
conn = duckdb.connect()

/tmp/ipykernel_6356/1239539535.py:1: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(json.dumps(data["Time Series (Digital Currency Daily)"]), orient='index')


In [105]:
pdf_data = conn.execute("""
    SELECT CAST("date" AS DATE) AS dt,
        CAST("1. open" AS DOUBLE) AS open_price,
        CAST("2. high" AS DOUBLE) AS high_price,
        CAST("3. low" AS DOUBLE) AS low_price,
        CAST("4. close" AS DOUBLE) AS close_price,
        CAST("5. volume" AS DOUBLE) AS trading_volume,
        CURRENT_TIMESTAMP AS processed_at
    FROM btc_data
""").fetchdf()

In [106]:
pdf_data.head()

,dt,open_price,high_price,low_price,close_price,trading_volume,processed_at
0,2026-03-07,68108.23,68383.71,68100.01,68209.06,181.139200,2026-03-07 16:51:20.372233+00:00
1,2026-03-06,70887.61,71428.40,67725.14,68108.24,11045.214396,2026-03-07 16:51:20.372233+00:00
2,2026-03-05,72683.26,73588.24,70639.01,70887.60,10891.718261,2026-03-07 16:51:20.372233+00:00
3,2026-03-04,68335.99,74100.00,67391.17,72683.26,21142.770792,2026-03-07 16:51:20.372233+00:00
4,2026-03-03,68832.00,69255.36,66138.20,68335.99,12823.727595,2026-03-07 16:51:20.372233+00:00


In [12]:
pd.set_option("max_colwidth", 1000)